In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import partial

# ==========================================
# 1. 初期設定
# ==========================================
catalog_name = 'suda_catalog'
test_schema_name = 'test'
result_schema_name = 'result'
setting_schema_name = 'setting'
setting_table_name = 'pk_master'
error_log_table_path = f"{catalog_name}.{setting_schema_name}.error_log"
checkpoint_root_path = "s3://my-company-uc-metastore-bucket/cdc_pipeline/" 

# 異常データ検出時の動作設定 (False: 検知時に処理完了後エラーで停止)
SKIP_BAD_DATA_MODE = False  

# 設定テーブルの取得
tables_df = spark.sql(f"SHOW TABLES IN {catalog_name}.{test_schema_name}")
existing_tables = [row["tableName"] for row in tables_df.collect()]
setting = spark.read.table(f"{catalog_name}.{setting_schema_name}.{setting_table_name}").collect()

# ==========================================
# 2. マイクロバッチ処理関数 (foreachBatch)
# ==========================================
def process_cdc_batch(new_df, batch_id, base_table_param, primary_keys_param, log_type_param, 
                      target_table_path_param, error_log_table_path_param):
    if new_df.isEmpty():
        return
    
    spark = new_df.sparkSession
    new_df_with_errors = new_df
    
    # --- (A) エラー条件の付与 ---
    new_df_with_errors = new_df_with_errors.withColumn(
        "error_reason",
        F.when(F.col("cdc_primekey").isNull(), F.lit("cdc_primekey_missing"))
         .when(F.col("op_type").isNull(), F.lit("op_type_missing"))
         .when(F.col("table").isNull(), F.lit("table_name_missing"))
         .when(~F.col("op_type").isin("I", "U", "D"), F.lit("unknown_op_type"))
         .when((F.col("op_type") == "I") & F.col("after").isNull(), F.lit("after_payload_missing"))
         .when((F.col("op_type") == "U") & F.col("before").isNull(), F.lit("before_payload_missing"))
         .when((F.col("op_type") == "U") & F.col("after").isNull(), F.lit("after_payload_missing"))
         .when((F.col("op_type") == "D") & F.col("before").isNull(), F.lit("before_payload_missing"))
    )
    for pk in primary_keys_param:
        is_null_cond = F.coalesce(F.col("after").getItem(pk), F.col("before").getItem(pk)).isNull()
        new_df_with_errors = new_df_with_errors.withColumn(
            "error_reason",
            F.when(F.col("error_reason").isNull() & is_null_cond, F.lit("primary_key_missing"))
             .otherwise(F.col("error_reason"))
        )

    # --- (B) DLQ（エラー退避）---
    bad_df = new_df_with_errors.filter(F.col("error_reason").isNotNull())
    bad_count = bad_df.count()
    if bad_count > 0:
        print(f"異常データ検知({base_table_param}): {bad_count}件をエラーログに退避します。")
        error_log_df = bad_df.withColumn("target_table", F.lit(target_table_path_param)) \
                             .withColumn("detected_at", F.current_timestamp())
        error_log_df.write.format("delta").mode("append").saveAsTable(error_log_table_path_param)
        
    batch_df = new_df_with_errors.filter(F.col("error_reason").isNull())
    if batch_df.isEmpty():
        return

    # --- (C) 正常データの処理（カラム展開） ---
    before_key_rows = batch_df.filter(F.col("before").isNotNull()).selectExpr("explode(map_keys(before)) as key").distinct().collect()
    after_key_rows = batch_df.filter(F.col("after").isNotNull()).selectExpr("explode(map_keys(after)) as key").distinct().collect()
    target_columns = sorted(list(set([row["key"] for row in before_key_rows] + [row["key"] for row in after_key_rows])))

    columns_definition = ",\n        ".join([f"{col} STRING" for col in target_columns])
    if log_type_param == 2:
        columns_definition += ",\n        is_latest INT,\n        exe_at TIMESTAMP"

    # テーブル作成
    create_table = f"""
        CREATE TABLE IF NOT EXISTS {target_table_path_param} (
            {columns_definition},
            inserted_at TIMESTAMP 
        ) USING DELTA TBLPROPERTIES ('delta.enableDeletionVectors' = true)
    """
    spark.sql(create_table)
    target_table = DeltaTable.forName(spark, target_table_path_param)
    
    parsed_df = batch_df
    for col_name in target_columns:
        parsed_df = parsed_df.withColumn(col_name, F.coalesce(F.col("after")[col_name], F.col("before")[col_name]))

    # inserted_atの降順で並び替え（バッチ内の最新を取得）
    window_spec = Window.partitionBy(*primary_keys_param).orderBy(F.col("inserted_at").desc())
    parsed_df = parsed_df.withColumn("rn", F.row_number().over(window_spec))

    # --- (D) MERGE実行（Last-Write-Wins 防波堤） ---
    if log_type_param == 1:
        latest_cdc_df = parsed_df.filter(F.col("rn") == 1).drop("rn")
        merge_condition = " AND ".join([f"target.{pk} = source.{pk}" for pk in primary_keys_param])
        
        update_mapping = {col: f"source.{col}" for col in target_columns if col not in primary_keys_param}
        update_mapping["inserted_at"] = "source.inserted_at"
        
        insert_mapping = {col: f"source.{col}" for col in target_columns}
        insert_mapping.update({pk: f"source.{pk}" for pk in primary_keys_param})
        insert_mapping["inserted_at"] = "source.inserted_at"

        (target_table.alias("target")
            .merge(latest_cdc_df.alias("source"), merge_condition)
            # 過去のデータによる上書き（順序逆転）をブロック
            .whenMatchedDelete(condition="source.op_type = 'D' AND source.inserted_at >= target.inserted_at")
            .whenMatchedUpdate(condition="source.op_type IN ('I', 'U') AND source.inserted_at >= target.inserted_at", set=update_mapping)
            .whenNotMatchedInsert(condition="source.op_type IN ('I', 'U')", values=insert_mapping)
            .execute()
        )
    
    elif log_type_param == 2:
        # (1) ターゲットテーブルに現在いる「最新レコード(is_latest=1)」の時刻を取得
        target_latest_df = spark.read.table(target_table_path_param).filter(F.col("is_latest") == 1).select(
            *[F.col(pk).alias(f"tgt_{pk}") for pk in primary_keys_param],
            F.col("inserted_at").alias("tgt_inserted_at")
        )
        
        # (2) 届いたバッチデータと結合
        join_cond = [F.col(pk) == F.col(f"tgt_{pk}") for pk in primary_keys_param]
        joined_df = parsed_df.join(target_latest_df, join_cond, "left")

        # (3) バッチ内最新(rn=1) かつ (ターゲットに未存在 OR ターゲットの時刻より新しい/同じ) 場合のみ 1
        parsed_batch_df = joined_df.withColumn(
            "is_latest",
            F.when(
                (F.col("rn") == 1) & (F.col("tgt_inserted_at").isNull() | (F.col("inserted_at") >= F.col("tgt_inserted_at"))),
                1
            ).otherwise(0)
        ).drop(*[f"tgt_{pk}" for pk in primary_keys_param], "tgt_inserted_at", "rn")

        # (4) MERGE実行（既存の1を0に降格）
        update_condition = " AND ".join([f"target.{pk} = source.{pk}" for pk in primary_keys_param])
        
        (target_table.alias("target")
            .merge(
                # source側は「新しく 1 になるべきレコード」だけを降格トリガーとして使う
                parsed_batch_df.filter(F.col("is_latest") == 1).alias("source"),
                update_condition
            )
            .whenMatchedUpdate(
                condition="target.is_latest = 1",
                set={"is_latest": "0"} 
            )
            .execute()
        )
        
        # (5) データの追加（APPEND）
        append_df = parsed_batch_df.filter(F.col("op_type").isin("I", "U"))
        columns_to_insert = target_columns + ["inserted_at", "is_latest", F.current_timestamp().alias("exe_at")]
        
        append_df.select(*columns_to_insert) \
                .write \
                .format("delta") \
                .mode("append") \
                .saveAsTable(target_table_path_param)


# ==========================================
# 3. メインループ実行
# ==========================================
for row in setting:
    base_table = row["table"]
    primary_keys = row["primary_key"]
    log_type = row["type"]

    if base_table not in existing_tables:
        continue

    print(f"\n======処理開始:{base_table}======")
    source_table_path = f"{catalog_name}.{test_schema_name}.{base_table}"
    target_table_path = f"{catalog_name}.{result_schema_name}.{base_table}_result"
    table_checkpoint_path = f"{checkpoint_root_path}/{base_table}"

    # 処理前のエラー件数取得
    error_count_before = 0
    if spark.catalog.tableExists(error_log_table_path):
        error_count_before = spark.read.table(error_log_table_path).filter(F.col("target_table") == target_table_path).count()

    # ストリーミング実行
    process_wrapper = partial(
        process_cdc_batch, base_table_param=base_table, primary_keys_param=primary_keys, 
        log_type_param=log_type, target_table_path_param=target_table_path, error_log_table_path_param=error_log_table_path
    )
    
    query = (spark.readStream
        .option("skipChangeCommits", "true") 
        .table(source_table_path)
        .writeStream
        .foreachBatch(process_wrapper)
        .option("checkpointLocation", table_checkpoint_path)
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination() 

    # 処理後のジョブ停止判定
    error_count_after = spark.read.table(error_log_table_path).filter(F.col("target_table") == target_table_path).count()
    if error_count_after > error_count_before:
        if not SKIP_BAD_DATA_MODE:
            raise RuntimeError(f"異常終了: {base_table} で異常データが検出されました。エラーログを確認し修正してください。")
        else:
            print(f" [SKIP MODE] 異常を検知しましたが処理を継続しました。")


======処理開始:table_c======

======処理開始:table_b======

======処理開始:table_f======

======処理開始:table_d======

======処理開始:table_a======

======処理開始:table_e======
